# Always-on bot vs wake-word daemon (stage 1)

Compares two ways of starting Jarvie with a stage-1 phrase ("jarvie, wake up", ...):

| | always-on bot | daemon |
|---|---|---|
| listening | `bot.py` runs all day; Deepgram transcribes everything | only the local wake-word model |
| wake | transcript contains a stage-1 phrase | model score >= threshold, then `bot.py` starts |
| after wake | bot answers | bot runs until a 2-minute idle timeout |

**daemon + confirm** (two-stage): as daemon, but the bot first checks Deepgram's transcript of the buffered wake
audio for a real stage-1 phrase, and goes straight back to sleep if there isn't one. That filters near-misses the
model can't ("jacky wake up", "wakie wake") without retraining.

1. **Accuracy** on your labeled clips: Deepgram transcript matching vs the local model.
2. **Cost** per day and month: speech-to-text minutes, LLM tokens, dollars.

Stage 2 (the short "jarvie" while the bot is awake) is the same in both designs and left out.
Kernel: `server/.venv` (pipecat-quickstart). Needs `DEEPGRAM_API_KEY`, loaded from `server/.env` like `bot.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

SERVER = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models").is_dir() and (p / "jarvie").is_dir())
sys.path.insert(0, str(SERVER))

from jarvie.eval import compare, gating, wakeword  # noqa: E402

load_dotenv(SERVER / ".env", override=True)  # same as bot.py: DEEPGRAM_API_KEY
pd.set_option("display.precision", 3)

## Settings
- `THRESHOLD`: the daemon's detection threshold. Use the EER threshold from `wakeword_eval.ipynb` (section 5).
- `STAGE1_PHRASES` / `JARVIE_SPELLINGS`: what counts as a wake phrase in a Deepgram transcript. Deepgram
  doesn't know "jarvie", so it may write Jarvis, Jarvy, Javi, ...; section 1 shows what it actually hears.
- `ASSUMPTIONS`: your usage and current prices. Everything in the cost section comes from here.

In [ ]:
AUDIO_DIR = Path.home() / "jarvie-data" / "audio" / "wake"
MODEL = SERVER / "models" / "jarvie_wake__B__base.onnx"
THRESHOLD = 0.89  # B's EER threshold in compare_models.ipynb (with the exclusions below)

# Same labeling as compare_models.ipynb: leave out "jarvie, I need some help" (B wasn't trained on it),
# and score acceptable wake phrases ("jarvie wakie wake", maybe "jarvis wake up") as positives.
EXCLUDE = {f"pos_desk_loud-normal_wake_{i:03d}.wav" for i in range(4, 8)}
AS_POSITIVE: set[str] = set()
TRANSCRIPT_CACHE = Path.home() / "jarvie-data" / "eval_cache" / "deepgram"
BOT_LOG = None  # e.g. SERVER / "bot.log" from `uv run bot.py 2>&1 | tee bot.log`, for measured tokens

STAGE1_PHRASES = gating.STAGE1_PHRASES
JARVIE_SPELLINGS = gating.JARVIE_SPELLINGS
PATTERN = gating.phrase_pattern(STAGE1_PHRASES, JARVIE_SPELLINGS)

ASSUMPTIONS = gating.Assumptions(
    listening_hours_per_day=12,
    activations_per_day=5,
    idle_timeout_minutes=2,
    conversation_minutes=1.5,
)
gating.assumptions_table(ASSUMPTIONS).to_frame("value")

## 1. What Deepgram hears
Each clip is sent once; transcripts are cached in `~/jarvie-data/eval_cache/deepgram/` by file content.

In [ ]:
session, input_name, window = wakeword.load_model(MODEL)
scored = gating.add_labels(wakeword.score_all(session, AUDIO_DIR, input_name, window, 1, wakeword.make_featurizer()))
scored = compare.apply_labels(scored, EXCLUDE, AS_POSITIVE)
files = [AUDIO_DIR / f for f in scored.file]

transcribe = gating.deepgram_transcriber(os.environ["DEEPGRAM_API_KEY"])
transcripts = gating.transcribe_clips(files, transcribe, TRANSCRIPT_CACHE)

decisions = gating.compare_detections(scored, transcripts, THRESHOLD, PATTERN)
decisions[["file", "kind", "pace", "similarity", "transcript", "always_on_fires", "max_score", "daemon_fires"]]

## 2. Accuracy: always-on (Deepgram + phrase match) vs daemon (local model) vs daemon + confirm
Recall on positives per pace, and how often each design fires on negatives per similarity tier. Clip counts
are small, so differences under ~20 points are noise.

In [ ]:
accuracy = gating.accuracy_table(decisions)
accuracy

In [ ]:
# Clips where exactly one design fires: listen to these.
gating.disagreements(decisions)

## 3. Tokens per activation
From a bot log if `BOT_LOG` is set (Pipecat logs `prompt tokens: N, completion tokens: M` per LLM call;
the first call is the startup greeting), otherwise the estimates in `ASSUMPTIONS`.

In [ ]:
assumptions = ASSUMPTIONS
if BOT_LOG and Path(BOT_LOG).exists():
    usage = gating.parse_token_usage(Path(BOT_LOG).read_text())
    print(f"{len(usage)} LLM calls in {BOT_LOG}")
    display(usage.describe().loc[["count", "mean", "min", "max"]])
    assumptions = assumptions.with_measured_tokens(usage)
else:
    print("no bot log: using the token estimates in ASSUMPTIONS")

# Recall measured above, on normal-pace positives (what you'd say on purpose).
normal = decisions[(decisions.kind == "pos") & (decisions.pace == "normal")]
confirmation = gating.estimate_confirmation(decisions)
assumptions = replace(
    assumptions,
    daemon_recall=float(normal.daemon_fires.mean()),
    always_on_recall=float(normal.always_on_fires.mean()),
    **confirmation,
)
print(f"recall used: daemon {assumptions.daemon_recall:.1%}, always-on {assumptions.always_on_recall:.1%}, "
      f"daemon + confirm {assumptions.confirmed_recall:.1%}")
print(f"false daemon starts that survive the transcript check: {assumptions.confirm_pass_rate:.0%} (from the clips)")

## 4. Cost per day
- **always-on** pays Deepgram for every listening minute; the LLM only runs on wakes.
- **daemon** pays Deepgram and the LLM only while the bot is up. Each start (real or false) costs a greeting
  and keeps the bot up until the idle timeout, so its false activations cost more than the always-on bot's.
- **daemon_confirmed** spends `confirm_seconds` of Deepgram on every start; only starts whose transcript has a
  stage-1 phrase greet you and wait for the idle timeout.
- `attempts_per_activation` = 1 / recall: how many times you'd say the phrase on average before it wakes.

In [ ]:
costs = gating.daily_costs(assumptions)
costs[["stt_minutes", "total_tokens", "false_activations", "attempts_per_activation", "stt_usd", "llm_usd", "total_usd"]]

In [ ]:
s = gating.savings(assumptions)
s_confirmed = gating.savings(assumptions, "daemon_confirmed")
pd.DataFrame({"daemon vs always-on": s, "daemon + confirm vs always-on": s_confirmed})

## 5. Sensitivity
The daemon's false activations per hour are the biggest unknown (a few minutes of negative clips can't measure
them). How savings change with them, and with listening hours:

In [ ]:
sweep = gating.sweep_false_activations(assumptions, [0, 0.1, 0.3, 0.5, 1, 2, 5])
sweep_confirmed = gating.sweep_false_activations(assumptions, [0, 0.1, 0.3, 0.5, 1, 2, 5], "daemon_confirmed")
display(sweep[["usd_per_month_saved", "pct_cost_saved", "stt_minutes_per_day_saved", "tokens_per_day_saved"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(sweep.index, sweep.usd_per_month_saved, marker="o", label="daemon")
axes[0].plot(sweep_confirmed.index, sweep_confirmed.usd_per_month_saved, marker="o", label="daemon + confirm")
axes[0].legend()
axes[0].set(title="Monthly savings vs daemon false activations", xlabel="daemon false activations per hour",
            ylabel="USD saved per month")
hours = np.arange(1, 25)
monthly = [gating.savings(replace(assumptions, listening_hours_per_day=h))["usd_per_month_saved"] for h in hours]
axes[1].plot(hours, monthly, marker=".")
axes[1].set(title="Monthly savings vs listening hours", xlabel="listening hours per day", ylabel="USD saved per month")
for ax in axes:
    ax.axhline(0, color="gray", lw=1)
plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
ao, dm, cf = costs.loc["always_on"], costs.loc["daemon"], costs.loc["daemon_confirmed"]
print(f"Per month: always-on ${30 * ao.total_usd:.2f}, daemon ${30 * dm.total_usd:.2f} "
      f"-> save ${s['usd_per_month_saved']:.2f} ({s['pct_cost_saved']:.0f}%)")
print(f"Speech-to-text: {ao.stt_minutes:.0f} vs {dm.stt_minutes:.0f} min/day; "
      f"tokens: {ao.total_tokens:,.0f} vs {dm.total_tokens:,.0f} per day")
print(f"Per month with confirmation: ${30 * cf.total_usd:.2f} -> save ${s_confirmed['usd_per_month_saved']:.2f}; "
      f"false wakes per day {dm.false_activations:.1f} -> {cf.false_activations:.1f}")
print(f"Recall (normal pace): always-on {assumptions.always_on_recall:.1%}, daemon {assumptions.daemon_recall:.1%}, "
      f"daemon + confirm {assumptions.confirmed_recall:.1%}")